# Nemotron Leaderboard Engine

Use this notebook as the Kaggle-side competition notebook for the NVIDIA Nemotron Model Reasoning Challenge.

Before running:
- Attach the competition input.
- Attach the Kaggle model `metric/nemotron-3-nano-30b-a3b-bf16/Transformers/default`.
- Set `Settings -> Accelerator -> GPU RTX Pro 6000` first.
- Keep internet on only long enough to upgrade PyTorch if the Blackwell cell tells you to.

Run strategy:
1. Run the environment and input inspection cells.
2. Run the competition/model discovery cells.
3. Run the Blackwell compatibility cell. If it upgrades PyTorch, the kernel will restart automatically.
4. After restart, rerun from the top until you reach the tokenizer cell.
5. Run the tokenizer cell.
6. Run the full model load cell.
7. If the model loads, run the short generation smoke test.
8. Only then build the real competition inference loop.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import torch

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
COMP_DIR = Path('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge')

print('Torch version:', torch.__version__)
print('Torch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('Input root exists:', INPUT_ROOT.exists())
print('Working root:', WORK_ROOT)

GPU_NAME = None
GPU_CAPABILITY = None
BLACKWELL_GPU = False

if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f'GPU {index}: {torch.cuda.get_device_name(index)}')
        print('  total_memory_gb =', round(props.total_memory / 1024**3, 2))

    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_CAPABILITY = torch.cuda.get_device_capability(0)
    BLACKWELL_GPU = 'blackwell' in GPU_NAME.lower() or GPU_CAPABILITY[0] >= 12
    print('GPU capability:', GPU_CAPABILITY)
    print('Blackwell GPU detected:', BLACKWELL_GPU)
else:
    print('No CUDA GPU detected. Stop here and turn on a GPU before loading the model.')

In [ ]:
top_level_inputs = sorted(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
print('Top-level /kaggle/input entries:')
for path in top_level_inputs:
    print('-', path)

sample_submission_candidates = sorted(INPUT_ROOT.rglob('sample_submission.csv'))
test_like_candidates = sorted(
    path for path in INPUT_ROOT.rglob('*')
    if path.name.lower() in {'test.csv', 'test.parquet', 'test.jsonl'}
)

print('\nSample submission candidates:')
for path in sample_submission_candidates[:10]:
    print('-', path)

print('\nTest file candidates:')
for path in test_like_candidates[:10]:
    print('-', path)

In [ ]:
def find_model_candidates(search_root: Path) -> list[Path]:
    candidates = []
    for config_path in search_root.rglob('config.json'):
        parent = config_path.parent
        if (parent / 'tokenizer_config.json').exists():
            candidates.append(parent)

    return sorted(
        set(candidates),
        key=lambda path: ('nemotron' not in str(path).lower(), len(path.parts), str(path)),
    )


MODEL_CANDIDATES = find_model_candidates(INPUT_ROOT)
print('Model candidates:')
for index, path in enumerate(MODEL_CANDIDATES):
    print(index, path)

assert MODEL_CANDIDATES, 'No model root detected under /kaggle/input.'
MODEL_DIR = MODEL_CANDIDATES[0]
print('\nSelected MODEL_DIR =', MODEL_DIR)

SAMPLE_SUBMISSION_PATH = sample_submission_candidates[0] if sample_submission_candidates else None
TEST_DATA_PATH = test_like_candidates[0] if test_like_candidates else None
print('SAMPLE_SUBMISSION_PATH =', SAMPLE_SUBMISSION_PATH)
print('TEST_DATA_PATH =', TEST_DATA_PATH)

In [ ]:
print('Competition files:')
if COMP_DIR.exists():
    for path in sorted(COMP_DIR.rglob('*')):
        if path.is_file():
            print('-', path)
else:
    print('Competition directory not found:', COMP_DIR)

TEST_COLUMNS = []
if TEST_DATA_PATH is not None:
    test_preview = pd.read_csv(TEST_DATA_PATH, nrows=5)
    TEST_COLUMNS = list(test_preview.columns)
    print('\ntest.csv columns:', TEST_COLUMNS)
    display(test_preview)
else:
    print('TEST_DATA_PATH is not available.')

## Blackwell Compatibility Fix

Kaggle's default PyTorch build may not support the Blackwell GPU architecture yet.

This cell will:
- detect that mismatch,
- install a CUDA 12.8-compatible PyTorch build if needed,
- write a marker file into `/kaggle/working`,
- and restart the kernel automatically.

If the kernel restarts, simply rerun from the top.

In [ ]:
import subprocess
import sys

TORCH_UPGRADE_MARKER = WORK_ROOT / '.torch_blackwell_ready'

def parse_cuda_version(raw: str | None) -> tuple[int, int]:
    if not raw:
        return (0, 0)
    parts = raw.split('.')
    major = int(parts[0]) if parts else 0
    minor = int(parts[1]) if len(parts) > 1 else 0
    return (major, minor)


def install_blackwell_torch() -> None:
    attempts = [
        (
            'stable-cu128',
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--upgrade',
                'torch',
                'torchvision',
                'torchaudio',
                '--index-url',
                'https://download.pytorch.org/whl/cu128',
            ],
        ),
        (
            'nightly-cu128',
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '--quiet',
                '--upgrade',
                '--pre',
                'torch',
                'torchvision',
                'torchaudio',
                '--index-url',
                'https://download.pytorch.org/whl/nightly/cu128',
            ],
        ),
    ]

    last_error = None
    for label, command in attempts:
        print(f'Trying torch install target: {label}')
        try:
            subprocess.check_call(command)
            print(f'Succeeded with {label}')
            return
        except subprocess.CalledProcessError as exc:
            last_error = exc
            print(f'Failed with {label}: return code {exc.returncode}')

    raise RuntimeError(
        'Unable to install a Blackwell-compatible torch build. Turn Kaggle internet on, rerun this cell, and only continue once the kernel restarts cleanly.'
    ) from last_error


current_cuda_build = parse_cuda_version(torch.version.cuda)
needs_blackwell_upgrade = bool(torch.cuda.is_available() and BLACKWELL_GPU and current_cuda_build < (12, 8))

print('Current CUDA build:', current_cuda_build)
print('Needs Blackwell torch upgrade:', needs_blackwell_upgrade)
print('Upgrade marker exists:', TORCH_UPGRADE_MARKER.exists())

if needs_blackwell_upgrade and not TORCH_UPGRADE_MARKER.exists():
    print('Installing a Blackwell-compatible torch build...')
    install_blackwell_torch()
    TORCH_UPGRADE_MARKER.write_text('upgraded\n', encoding='utf-8')
    print('Torch upgrade complete. Restarting kernel now. After restart, rerun from the top.')
    os.kill(os.getpid(), 9)
elif needs_blackwell_upgrade:
    print('Blackwell upgrade marker found. Continue with the next cells.')
else:
    print('Current torch build is acceptable for this accelerator. Continue with the next cells.')

## Load The Tokenizer First

This should be cheap. If this fails, the attached model path is wrong or the Kaggle model input is incomplete.

In [ ]:
assert 'MODEL_DIR' in globals(), 'Run the model discovery cell first.'

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
print('Tokenizer loaded from:', MODEL_DIR)
print('Tokenizer class:', tokenizer.__class__.__name__)
print('Tokenizer vocab size:', getattr(tokenizer, 'vocab_size', 'unknown'))

## Load The Model

This is the expensive step. If it fails with OOM, stop the session and do not keep retrying blindly.

In [ ]:
assert 'MODEL_DIR' in globals(), 'Run the model discovery cell first.'
assert 'tokenizer' in globals(), 'Run the tokenizer cell first.'

TORCH_UPGRADE_MARKER = WORK_ROOT / '.torch_blackwell_ready'
current_cuda_build = parse_cuda_version(torch.version.cuda)
assert not (torch.cuda.is_available() and BLACKWELL_GPU and current_cuda_build < (12, 8) and not TORCH_UPGRADE_MARKER.exists()), (
    'Run the Blackwell compatibility cell and let the kernel restart before loading the model.'
)

from transformers import AutoModelForCausalLM

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    torch_dtype=dtype,
    trust_remote_code=True,
    device_map='auto',
    low_cpu_mem_usage=True,
)
model.eval()

def model_input_device(model):
    if hasattr(model, 'device'):
        return model.device
    return next(iter(model.parameters())).device


print('Model loaded successfully.')
print('Input device:', model_input_device(model))
print('Device map:', getattr(model, 'hf_device_map', 'single-device'))

In [ ]:
assert 'model' in globals(), 'Run the full model load cell first.'

def tokenize_messages(messages, enable_thinking=None):
    kwargs = {
        'tokenize': True,
        'add_generation_prompt': True,
        'return_tensors': 'pt',
    }
    if enable_thinking is not None:
        try:
            return tokenizer.apply_chat_template(messages, enable_thinking=enable_thinking, **kwargs)
        except TypeError:
            print('Tokenizer does not expose enable_thinking; retrying without it.')
    return tokenizer.apply_chat_template(messages, **kwargs)


messages = [
    {
        'role': 'user',
        'content': 'Solve briefly: what is 17 * 19? Return only the final answer.',
    }
]

inputs = tokenize_messages(messages, enable_thinking=False).to(model_input_device(model))

with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=32,
        do_sample=False,
        num_beams=1,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_tokens = outputs[0][inputs.shape[-1]:]
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print(generated_text)

In [ ]:
assert 'model' in globals(), 'Run the full model load cell first.'

RUN_THINKING_TEST = False

if RUN_THINKING_TEST:
    thinking_inputs = tokenize_messages(messages, enable_thinking=True).to(model_input_device(model))
    with torch.no_grad():
        thinking_outputs = model.generate(
            thinking_inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=1.0,
            top_p=1.0,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_tokens = thinking_outputs[0][thinking_inputs.shape[-1]:]
    print(tokenizer.decode(generated_tokens, skip_special_tokens=False))
else:
    print('Set RUN_THINKING_TEST = True only after the short no-thinking smoke test succeeds.')

In [ ]:
if SAMPLE_SUBMISSION_PATH is not None:
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    print('Sample submission shape:', sample_submission.shape)
    print('Columns:', list(sample_submission.columns))
    display(sample_submission.head())

    prediction_columns = [
        column for column in sample_submission.columns
        if column.lower() not in {'id', 'index'}
    ]

    if prediction_columns:
        submission = sample_submission.copy()
        submission[prediction_columns[0]] = ''
        submission_path = WORK_ROOT / 'submission.csv'
        submission.to_csv(submission_path, index=False)
        print('Draft submission scaffold written to:', submission_path)
        display(submission.head())
    else:
        print('Could not infer the prediction column from the sample submission.')
elif TEST_DATA_PATH is not None and TEST_COLUMNS:
    test_id_col = next((column for column in TEST_COLUMNS if column.lower() in {'id', 'index'}), TEST_COLUMNS[0])
    test_ids = pd.read_csv(TEST_DATA_PATH, usecols=[test_id_col])
    placeholder_submission = pd.DataFrame({test_id_col: test_ids[test_id_col], 'answer': ''})
    placeholder_path = WORK_ROOT / 'submission_placeholder.csv'
    placeholder_submission.to_csv(placeholder_path, index=False)
    print('No sample submission was attached. Wrote a provisional placeholder to:', placeholder_path)
    display(placeholder_submission.head())
    print('Replace the provisional answer column name after the official submission schema is confirmed.')
else:
    print('Neither sample submission nor test columns are available.')

## Next Step

If the model loads cleanly and the smoke test works on the selected GPU, replace the toy prompt cell with the real competition inference loop and build predictions from the competition test input.